# Avatar_ML Colab inference server

Runs the **data plane** of the hybrid system: OpenVoice V2 + MuseTalk 1.5 behind a FastAPI server exposed through a Cloudflare Tunnel. Windows talks to this URL.

## How to use

1. **Runtime → Change runtime type → T4 GPU**.
2. Run all cells, top to bottom. Cell 2 takes 6–10 minutes (it installs Python 3.11 and a lot of ML deps).
3. Copy the printed `COLAB_INFERENCE_URL=https://*.trycloudflare.com` line into your local `.env` on Windows.
4. Keep this tab open — closing it idles the runtime quickly.

When Colab disconnects, restart and re-run all cells, paste the new URL into Windows `.env`, then run `python scripts/create_avatar_profile.py --rehydrate <avatar_id>` on Windows.

## Why this notebook installs Python 3.11 separately

Colab's default kernel is Python **3.12**, but OpenVoice / MeloTTS / parts of MuseTalk's `requirements.txt` fail to build on 3.12 (their `setup.py` was written for 3.10/3.11). We install Python 3.11 with `apt` + `deadsnakes`, create a venv at `/content/py311`, install the model stack there, and run `uvicorn` from that venv. The notebook kernel itself stays on 3.12 — it just orchestrates.

## Two model-inference TODOs

The server code (Cell 3) includes the full wire protocol exactly as Windows expects. Two endpoints — `/avatar/preprocess` and `WS /lipsync/stream` — are marked **TODO** because MuseTalk's internal API varies by version. Each TODO points to the exact MuseTalk file to adapt. The other endpoints — `/healthz`, `/voice/extract_embedding`, `/avatar/upload_cache`, `WS /tts/stream` — are fully implemented.

In [ ]:
# Cell 2 — install Python 3.11, create venv, install model deps, download weights.
import os, sys, subprocess
print('Notebook kernel:', sys.version)
!nvidia-smi | head -n 4

# 1) Install Python 3.11 if missing. Colab base images vary; try direct apt
# first, fall back to the deadsnakes PPA.
have_311 = subprocess.run(['which', 'python3.11'], capture_output=True).returncode == 0
if not have_311:
    print('Installing Python 3.11...')
    !sudo apt-get update -qq
    !sudo apt-get install -y -qq software-properties-common
    !sudo add-apt-repository -y ppa:deadsnakes/ppa
    !sudo apt-get update -qq
    !sudo apt-get install -y -qq python3.11 python3.11-venv python3.11-dev
!python3.11 --version

# 2) Create the venv if needed.
PY311 = '/content/py311'
if not os.path.exists(f'{PY311}/bin/python'):
    !python3.11 -m venv {PY311}

PY  = f'{PY311}/bin/python'
PIP = f'{PY311}/bin/pip'
!{PIP} install --quiet --upgrade pip

# 3) Server runtime + pinned numpy + torch for CUDA 12.1 (T4).
!{PIP} install --quiet "fastapi>=0.115" "uvicorn[standard]>=0.32" "python-multipart>=0.0.12" "websockets>=13" "Pillow>=10.4" "soundfile>=0.12" "librosa==0.9.1" "huggingface_hub>=0.24" "numpy==1.26.4"
!{PIP} install --quiet torch==2.1.2 torchaudio==2.1.2 torchvision==0.16.2 --index-url https://download.pytorch.org/whl/cu121

# 4) Clone the three model repos.
for repo, url in [
    ('OpenVoice', 'https://github.com/myshell-ai/OpenVoice'),
    ('MeloTTS',   'https://github.com/myshell-ai/MeloTTS'),
    ('MuseTalk',  'https://github.com/TMElyralab/MuseTalk'),
]:
    if not os.path.exists(f'/content/{repo}'):
        !git clone --depth 1 {url} /content/{repo}

# 5) OpenVoice + MeloTTS runtime deps (we skip `pip install -e .` because their
# setup.py is brittle; instead we use .pth files below to add the repos to
# the venv's sys.path).
!{PIP} install --quiet faster-whisper==0.10.0 pydub==0.25.1 wavmark==0.0.3 eng_to_ipa==0.0.2 inflect==7.0.0 unidecode==1.3.7 pypinyin==0.50.0 cn2an==0.5.22 jieba==0.42.1 langid==1.1.6 dominate==2.8.0 openai-whisper
!{PIP} install --quiet mecab-python3==1.0.9 unidic-lite==1.0.8 num2words==0.5.13 pykakasi==2.2.1 fugashi==1.3.2 g2p_en==2.1.0 anyascii==0.3.2 jamo==0.4.1 gruut==2.3.4 cached_path==1.6.2 nltk==3.8.1 "transformers==4.27.4" tokenizers==0.13.3

# 6) Make the cloned repos importable from the venv.
sp = f'{PY311}/lib/python3.11/site-packages'
!echo '/content/OpenVoice' > {sp}/openvoice.pth
!echo '/content/MeloTTS'   > {sp}/melo.pth
!echo '/content/MuseTalk'  > {sp}/musetalk.pth

# 7) MeloTTS needs unidic dictionary at runtime.
!{PY} -m unidic download 2>&1 | tail -3

# 8) Sanity-check the two TTS imports — this is the failure point we just fixed.
!{PY} -c "from openvoice.api import ToneColorConverter; print('OpenVoice ToneColorConverter: OK')"
!{PY} -c "from openvoice import se_extractor; print('OpenVoice se_extractor: OK')"
!{PY} -c "from melo.api import TTS; print('MeloTTS TTS: OK')"

# 9) Download OpenVoice V2 weights now — this is what VP2 needs. Doing this
# BEFORE any MuseTalk step so a MuseTalk failure can never block VP2.
OPENVOICE_CKPT = '/content/models/openvoice/checkpoints_v2'
os.makedirs(OPENVOICE_CKPT, exist_ok=True)
!{PY} -c "from huggingface_hub import snapshot_download; snapshot_download(repo_id='myshell-ai/OpenVoiceV2', local_dir='{OPENVOICE_CKPT}')"
print('OpenVoice weights ready. VP2 will work past this point.')

# 10) MuseTalk deps — OPTIONAL for VP2. Only needed once you finish the
# MuseTalk preprocessing/lipsync TODOs in /content/server.py.
# Each step is wrapped so a failure never aborts the cell. Specifically we
# do NOT install mmpose: it pulls in chumpy, which fails to build on Python
# 3.11. MuseTalk's lip-sync path only needs mmcv, not mmpose.
!{PIP} install --quiet openmim 2>&1 | tail -3 || echo '(openmim install failed; OK to skip for VP2)'
!{PY} -m mim install mmcv==2.1.0 2>&1 | tail -3 || echo '(mmcv install failed; finish manually before VP3)'
# MuseTalk's own requirements.txt has been observed to fail too; ignore for VP2.
!{PIP} install --quiet -r /content/MuseTalk/requirements.txt 2>&1 | tail -3 || echo '(MuseTalk requirements partial; OK to skip for VP2)'

# 11) MuseTalk weights (upstream uses .sh, not .py).
!cd /content/MuseTalk && bash download_weights.sh 2>&1 | tail -5 || echo '(MuseTalk weights skipped; finish manually before VP3)'

print('\n=== Cell 2 done: Python 3.11 venv ready at /content/py311 ===')
print('VP1 + VP2 are unblocked. VP3+ still need the MuseTalk TODOs in server.py.')

In [ ]:
%%writefile /content/server.py
"""Avatar_ML Colab inference server. Runs under /content/py311/bin/python.

Wire protocol mirrors Plan.md §5.5 — do not change message shapes without
also updating the Windows-side ColabWorkerClient.
"""
import os, sys, io, json, base64, time, tarfile, shutil, traceback
from pathlib import Path

import numpy as np
import soundfile as sf
import librosa
import torch
from fastapi import FastAPI, UploadFile, File, Request, Query, WebSocket, WebSocketDisconnect
from fastapi.responses import Response

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
OPENVOICE_CKPT = '/content/models/openvoice/checkpoints_v2'
MUSETALK_DIR  = '/content/MuseTalk'
CACHE_ROOT    = '/content/cache'
Path(CACHE_ROOT).mkdir(parents=True, exist_ok=True)
print('[server] Python:', sys.version.split()[0], '| Device:', DEVICE)

# ---- Lazy loaders ----------------------------------------------------------
_ov = {'tcc': None, 'melo': None, 'source_se': None}
_mt = {'loaded': False, 'modules': {}}

def lazy_load_openvoice():
    if _ov['tcc'] is not None:
        return _ov
    from openvoice.api import ToneColorConverter
    tcc = ToneColorConverter(f'{OPENVOICE_CKPT}/converter/config.json', device=DEVICE)
    tcc.load_ckpt(f'{OPENVOICE_CKPT}/converter/checkpoint.pth')
    _ov['tcc'] = tcc
    from melo.api import TTS
    _ov['melo'] = TTS(language='EN', device=DEVICE)
    _ov['source_se'] = torch.load(
        f'{OPENVOICE_CKPT}/base_speakers/ses/en-us.pth', map_location=DEVICE
    )
    print('[lazy] OpenVoice V2 + MeloTTS loaded.')
    return _ov

def lazy_load_musetalk():
    if _mt['loaded']:
        return _mt
    # VERIFY against the MuseTalk version in /content/MuseTalk/musetalk/utils/utils.py.
    from musetalk.utils.utils import load_all_model
    audio_processor, vae, unet, pe = load_all_model()
    vae.vae = vae.vae.half().to(DEVICE)
    unet.model = unet.model.half().to(DEVICE)
    pe = pe.half().to(DEVICE)
    _mt['modules'] = {
        'audio_processor': audio_processor, 'vae': vae, 'unet': unet, 'pe': pe,
    }
    _mt['loaded'] = True
    print('[lazy] MuseTalk loaded.')
    return _mt

app = FastAPI(title='Avatar_ML Colab inference server')

# ---- /healthz --------------------------------------------------------------
@app.get('/healthz')
def healthz():
    free_vram_mb = None
    if torch.cuda.is_available():
        free, _ = torch.cuda.mem_get_info()
        free_vram_mb = int(free / 1e6)
    return {
        'status': 'ok',
        'python': sys.version.split()[0],
        'gpu': torch.cuda.is_available(),
        'openvoice_loaded': _ov['tcc'] is not None,
        'musetalk_loaded': _mt['loaded'],
        'free_vram_mb': free_vram_mb,
    }

# ---- POST /voice/extract_embedding -----------------------------------------
@app.post('/voice/extract_embedding')
async def extract_voice_embedding(audio: UploadFile = File(...)):
    try:
        ov = lazy_load_openvoice()
        upload_dir = Path('/content/uploads/voice')
        upload_dir.mkdir(parents=True, exist_ok=True)
        wav_path = upload_dir / (audio.filename or 'voice.wav')
        wav_path.write_bytes(await audio.read())
        from openvoice import se_extractor
        target_se, _name = se_extractor.get_se(str(wav_path), ov['tcc'], vad=False)
        buf = io.BytesIO()
        np.save(buf, target_se.detach().cpu().numpy())
        return Response(content=buf.getvalue(), media_type='application/octet-stream')
    except Exception as e:
        traceback.print_exc()
        # Surface the real error to the Windows client instead of an opaque 500.
        from fastapi import HTTPException
        raise HTTPException(status_code=500, detail=f'{type(e).__name__}: {e}')

# ---- POST /avatar/preprocess -----------------------------------------------
@app.post('/avatar/preprocess')
async def preprocess_avatar(video: UploadFile = File(...)):
    """TODO: implement using MuseTalk preprocessing.

    Reference: /content/MuseTalk/scripts/realtime_inference.py, the Avatar
    class __init__ flow:
      1. extract frames from MP4,
      2. detect bboxes via get_landmark_and_bbox(),
      3. crop + resize each face to 256x256,
      4. VAE-encode each crop into a latent,
      5. save coords.pkl, latents.pt, masks/, idle frames into a dir.

    Then tar the dir and return as application/gzip.
    """
    from fastapi import HTTPException
    raise HTTPException(
        status_code=501,
        detail='MuseTalk preprocessing TODO. Adapt /content/MuseTalk/scripts/realtime_inference.py.',
    )

# ---- POST /avatar/upload_cache ---------------------------------------------
@app.post('/avatar/upload_cache')
async def upload_avatar_cache(request: Request, avatar_id: str = Query(...)):
    body = await request.body()
    dest = Path(CACHE_ROOT) / avatar_id
    if dest.exists():
        shutil.rmtree(dest)
    dest.mkdir(parents=True, exist_ok=True)
    tar_path = dest / 'cache.tar.gz'
    tar_path.write_bytes(body)
    with tarfile.open(tar_path, 'r:gz') as tar:
        tar.extractall(dest)
    return {'avatar_id': avatar_id, 'extracted_to': str(dest), 'size_bytes': len(body)}

# ---- helpers for TTS -------------------------------------------------------
def _to_24khz_int16(wav_path: str) -> bytes:
    data, sr = sf.read(wav_path, dtype='int16')
    if data.ndim > 1:
        data = data[:, 0]
    if sr != 24000:
        f = data.astype(np.float32) / 32768.0
        f = librosa.resample(f, orig_sr=sr, target_sr=24000)
        data = np.clip(f * 32768.0, -32768, 32767).astype(np.int16)
    return data.tobytes()

def _synthesize_pcm(text: str, target_se_b64: str) -> bytes:
    ov = lazy_load_openvoice()
    arr = np.load(io.BytesIO(base64.b64decode(target_se_b64)))
    target_se = torch.from_numpy(arr).to(DEVICE)
    if target_se.dim() == 1:
        target_se = target_se.unsqueeze(0).unsqueeze(-1)
    src_path = '/tmp/openvoice_src.wav'
    out_path = '/tmp/openvoice_out.wav'
    spk_ids = ov['melo'].hps.data.spk2id
    spk_key = 'EN-US' if 'EN-US' in spk_ids else 'EN-Default' if 'EN-Default' in spk_ids else next(iter(spk_ids))
    ov['melo'].tts_to_file(text, spk_ids[spk_key], src_path, speed=1.0)
    ov['tcc'].convert(
        audio_src_path=src_path,
        src_se=ov['source_se'],
        tgt_se=target_se,
        output_path=out_path,
    )
    return _to_24khz_int16(out_path)

# ---- WS /tts/stream --------------------------------------------------------
@app.websocket('/tts/stream')
async def tts_stream(ws: WebSocket):
    await ws.accept()
    try:
        payload = json.loads(await ws.receive_text())
        pcm_bytes = _synthesize_pcm(payload['text'], payload['embedding_b64'])
        # ~200 ms chunks: 24000 samples/s * 0.2 s * 2 bytes/sample = 9600 bytes.
        chunk_sz = 9600
        pts = 0.0
        for i in range(0, len(pcm_bytes), chunk_sz):
            chunk = pcm_bytes[i:i + chunk_sz]
            await ws.send_text(json.dumps({
                'type': 'pcm', 'pts': pts,
                'data': base64.b64encode(chunk).decode(),
            }))
            pts += (len(chunk) // 2) / 24000.0
        await ws.send_text(json.dumps({'type': 'end'}))
    except WebSocketDisconnect:
        pass
    except Exception as e:
        traceback.print_exc()
        try:
            await ws.send_text(json.dumps({'type': 'error', 'message': f'{type(e).__name__}: {e}'}))
        except Exception:
            pass

# ---- WS /lipsync/stream ----------------------------------------------------
@app.websocket('/lipsync/stream')
async def lipsync_stream(ws: WebSocket):
    """TODO: implement using MuseTalk inference.

    Reference: /content/MuseTalk/scripts/realtime_inference.py Avatar.inference():
      1. concatenate received PCM chunks into a WAV,
      2. audio_processor.audio2feat(wav_path) -> whisper features,
      3. for each (latent, audio_feat) batch: unet(...) -> decoded face crop,
      4. blend the 256x256 face back into the original frame via masks,
      5. JPEG-encode + send {'type':'frame', 'frame_idx':i, 'pts':i/25.0, 'jpeg_b64':...}.
    """
    await ws.accept()
    try:
        first = json.loads(await ws.receive_text())
        # Drain incoming PCM so the client doesn't deadlock waiting on us.
        while True:
            m = json.loads(await ws.receive_text())
            if m.get('type') == 'end':
                break
        await ws.send_text(json.dumps({
            'type': 'error',
            'message': 'MuseTalk lipsync TODO. See /content/MuseTalk/scripts/realtime_inference.py.',
        }))
    except WebSocketDisconnect:
        pass
    except Exception as e:
        traceback.print_exc()
        try:
            await ws.send_text(json.dumps({'type': 'error', 'message': f'{type(e).__name__}: {e}'}))
        except Exception:
            pass

if __name__ == '__main__':
    import uvicorn
    uvicorn.run(app, host='0.0.0.0', port=8000, log_level='info')


In [ ]:
# Cell 4 — install cloudflared and start the quick-tunnel.
import subprocess, re, threading, os

if not os.path.exists('/usr/local/bin/cloudflared'):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared

PORT = 8000
tunnel_url_ref = {'url': None}

def run_tunnel():
    proc = subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', f'http://localhost:{PORT}'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    for line in proc.stdout:
        print('[tunnel]', line, end='')
        m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
        if m and not tunnel_url_ref['url']:
            tunnel_url_ref['url'] = m.group(0)
            print(f'\n=== TUNNEL READY: {tunnel_url_ref["url"]} ===\n', flush=True)

threading.Thread(target=run_tunnel, daemon=True).start()
print('Starting Cloudflare quick-tunnel...')

In [ ]:
# Cell 5 — launch uvicorn from the Python 3.11 venv, print the public URL.
import subprocess, threading, time, sys

def run_uvicorn():
    proc = subprocess.Popen(
        ['/content/py311/bin/python', '-m', 'uvicorn',
         'server:app', '--host', '0.0.0.0', '--port', '8000'],
        cwd='/content',
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    for line in proc.stdout:
        print('[uvicorn]', line, end='')

threading.Thread(target=run_uvicorn, daemon=True).start()
print('Booting uvicorn from /content/py311/bin/python on /content/server.py ...')

deadline = time.time() + 180
while time.time() < deadline and not tunnel_url_ref.get('url'):
    time.sleep(1)

if tunnel_url_ref.get('url'):
    print()
    print('=' * 60)
    print(f'COLAB_INFERENCE_URL={tunnel_url_ref["url"]}')
    print('=' * 60)
    print('Copy the line above into your local .env on Windows.')
else:
    print('Tunnel URL not detected within 180s. Re-run Cell 4.')

## Finishing the two TODOs

Both stubs live in **`/content/server.py`** (written by Cell 3).

1. After Cell 2 finishes, open `/content/MuseTalk/scripts/realtime_inference.py` in Colab's file browser.
2. Locate the `Avatar` class. Its `__init__` does preprocessing; its `inference` method does the per-audio-chunk lip-sync.
3. Port the relevant logic into `preprocess_avatar` and `lipsync_stream` in `/content/server.py` (use `%%writefile -a /content/server.py` from a new cell to append, or edit the file directly via Colab's editor).
4. Restart uvicorn: stop Cell 5's running thread by interrupting the runtime (Runtime → Interrupt execution) and re-run Cells 4 and 5.

Once both stubs are real:

- `python scripts/create_avatar_profile.py samples/face.mp4` on Windows will succeed.
- `python scripts/generate_video.py --voice <id> --avatar <id> --text "..."` will produce an MP4.
- The browser test page will show the avatar speaking after clicking **Speak**.